# Importing Libraries

In [361]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder,PolynomialFeatures

# Checking the data again

In [362]:
df = pd.read_csv(r"C:\Skills\Student Performance Predictor\Dataset\Cleaned Dataset.csv")

In [363]:
df.head().T

,0,1,2,3,4
Unnamed: 0,0,1,2,3,4
University Admission year,2018,2021,2020,2021,2021
Gender,Male,Male,Female,Male,Male
Age,24,22,21,20,22
H.S.C passing year,2016,2020,2019,2020,2019
Current Semester,12,4,5,4,4
Do you have meritorious scholarship ?,Yes,Yes,No,Yes,Yes
Do you use University transportation?,No,Yes,No,No,No
How many hour do you study daily?,3,3,3,1,3
How many times do you seat for study in a day?,2,2,3,3,1


In [364]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [365]:
df.head().T

,0,1,2,3,4
University Admission year,2018,2021,2020,2021,2021
Gender,Male,Male,Female,Male,Male
Age,24,22,21,20,22
H.S.C passing year,2016,2020,2019,2020,2019
Current Semester,12,4,5,4,4
Do you have meritorious scholarship ?,Yes,Yes,No,Yes,Yes
Do you use University transportation?,No,Yes,No,No,No
How many hour do you study daily?,3,3,3,1,3
How many times do you seat for study in a day?,2,2,3,3,1
What is your preferable learning mode?,Offline,Online,Offline,Offline,Online


# Feature Engineering

In [366]:
x = df.drop(columns='What is your current CGPA?')
y = df['What is your current CGPA?']

In [367]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=100)

In [368]:
# Dividing Numerical Ordinal categorical and OneHot categorical in a list so we can easily preprocess
num_cols = x_train.describe().columns
one_hot_cols = x_train.drop(columns=num_cols).columns

In [369]:
ordinal_cols = ['Status of your English language proficiency']

In [370]:
one_hot_cols = one_hot_cols.drop(ordinal_cols)

In [371]:
# Now we have all columns seperated 

In [411]:
# Now making preprocessing pipelines
num_pipe = Pipeline([
    ('Impute',SimpleImputer(strategy='mean')),
    ('Normalization',PowerTransformer()),
    ('Scaling',StandardScaler()),
    ('Polynomial Features',PolynomialFeatures(degree=2))
])

ordinal_pipe = Pipeline([
    ('Impute',SimpleImputer(strategy='most_frequent')),
    ('Encode',OrdinalEncoder(categories=[['Basic','Intermediate','Advance']]))
])

one_hot_pipe = Pipeline([
    ('Impute',SimpleImputer(strategy='most_frequent')),
    ('Encode',OneHotEncoder(drop='first',sparse_output=False))
])

In [412]:
# Making preprocessing Collumn Transformer 
Preprocess = ColumnTransformer(transformers=[
    ('Numerical Preprocessing',num_pipe,num_cols),
    ('Ordinal Preprocessing',ordinal_pipe,ordinal_cols),
    ('One hot Preprocessing',one_hot_pipe,one_hot_cols)
],remainder='passthrough')

In [413]:
# Making the baseline model 

# Baseline Model making (Linear Regression)

In [414]:
from sklearn.linear_model import LinearRegression,Ridge,SGDRegressor,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

In [415]:
# Baseline
Model = Pipeline([
    ('Preprocessing',Preprocess),
    ('Model Training',LinearRegression())
])

Model.fit(x_train,y_train)
r2_score(y_test,Model.predict(x_test))

0.832446051672424

In [377]:
# So after removing some columns we are getting the r2 score of baseline as 72.88 % but now no removing we will test 

In [378]:
# Without any removal i am getting better results so i will keep all my columns for now and while production and deployment i can keep as the user input in my project

# Gradient Descent 

In [416]:
# Gradient Descent
Model = Pipeline([
    ('Preprocessing',Preprocess),
    ('Model Training',SGDRegressor(max_iter=1000,eta0=0.01))
])

Model.fit(x_train,y_train)
r2_score(y_test,Model.predict(x_test))

0.8121395378388041

In [380]:
# Performing almost same 

# Ridge

In [417]:
# Ridge Regression
Model = Pipeline([
    ('Preprocessing',Preprocess),
    ('Model Training',Ridge(alpha=60))
])

Model.fit(x_train,y_train)
r2_score(y_test,Model.predict(x_test))

0.8498252721783235

In [382]:
# Doing Hyper Parameter Tuning for my model 
param_grid = {
    "Model Training__alpha": [
        0.0001,
        0.001,
        0.01,
        0.1,
        1,
        10,
        11,
        15,
        20,
        25,
        30,
        40,
        50,
        60,
        70,
        100,
        1000
    ]
}

grid = GridSearchCV(
    estimator=Model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(x_train, y_train)
grid.best_params_

{'Model Training__alpha': 60}

In [383]:
# After doing Hyper Parameter tuning We can see our model is performing much better than the Normal Linear Regression it means our model was slightly overfitting

# Lasso

In [418]:
# Lasso Regression
Model = Pipeline([
    ('Preprocessing',Preprocess),
    ('Model Training',Lasso(alpha=0.01))
])

Model.fit(x_train,y_train)
r2_score(y_test,Model.predict(x_test))

0.8530354495778878

In [397]:
# Hyper Parameter Tuning 

param = {
    'Model Training__alpha' : [0.0001,0.001,0.005,0.002,0.003,0.004,0.01,0.006,0.007]
}

grid = GridSearchCV(
    estimator=Model,
    param_grid=param,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid.fit(x_train,y_train)
grid.best_params_

{'Model Training__alpha': 0.01}

In [386]:
# Best Performance Till now is given by our Ridge Now lets try elestic net because it uses both 

# Elastic Net

In [419]:
# Elastic Net Regression
Model = Pipeline([
    ('Preprocessing',Preprocess),
    ('Model Training',ElasticNet(alpha=0.01,l1_ratio=0.9))
])

Model.fit(x_train,y_train)
r2_score(y_test,Model.predict(x_test))

0.852718920038118

In [406]:
# Hyper Parameter Tuning 

param = {
    'Model Training__alpha' : [0.0001,0.001,0.005,0.002,0.003,0.004,0.01,0.006,0.007],
    'Model Training__l1_ratio':[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
}

grid = GridSearchCV(
    estimator=Model,
    param_grid=param,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid.fit(x_train,y_train)
grid.best_params_


{'Model Training__alpha': 0.01, 'Model Training__l1_ratio': 0.9}

In [399]:
# After Giving Polynomial Features to our Data we can reach 85.30 % r2 score which is really great (85.30%) 